In [0]:
from pyspark.sql import functions as F

fact_orders = (
    spark.read.table("databricks_catalog.silver_schema.orders")
    .select(
        "order_id",
        F.xxhash64("customer_id").alias("customer_key"),
        F.xxhash64("product_id").alias("product_key"),
        F.date_format("order_date", "yyyyMMdd").cast("int").alias("date_key"),
        "quantity",
        "total_amount",
        (F.col("total_amount") / F.col("quantity")).cast("decimal(12,2)").alias("unit_price"),
    )
    .withColumn("loaded_at", F.current_timestamp())
)

(fact_orders.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("databricks_catalog.gold_schema.fact_orders"))

print("✓ fact_orders written:", fact_orders.count(), "rows")

✓ fact_orders written: 10000 rows


In [0]:
g = "databricks_catalog.gold_schema"
f = spark.table(f"{g}.fact_orders")

print("Orders:", f.count())
print("Missing customer:", f.join(spark.table(f"{g}.dim_customer"), "customer_key", "left_anti").count())
print("Missing product: ", f.join(spark.table(f"{g}.dim_product"),  "product_key",  "left_anti").count())
print("Missing date:    ", f.join(spark.table(f"{g}.dim_date"),     "date_key",     "left_anti").count())

for t, k in [("dim_customer", "customer_key"), ("dim_product", "product_key"), ("dim_date", "date_key")]:
    d = spark.table(f"{g}.{t}")
    print(f"{t}: {d.count()} rows, {d.select(k).distinct().count()} unique keys")

Orders: 10000
Missing customer: 0
Missing product:  0
Missing date:     0
dim_customer: 2000 rows, 2000 unique keys
dim_product: 500 rows, 500 unique keys
dim_date: 731 rows, 731 unique keys
